# Análisis del indice de calor en Iquitos (1990 - 2025)

**Objetivo:** Identificar en que meses y horas el calor de Iquitos es mas concentrado para poder determinar la demanda de servicios de aire acondicionado.

**Fuente:** NCEI NOAA estacion Aeropuerto Internacional Crnl. FAP Francisco Secada Vignetta.

In [1]:
import pandas as pd

estaciones = pd.read_csv(
    "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv",
    dtype=str
)
estaciones[estaciones["ICAO"].str.contains("SPQT", na=False)]

,USAF,WBAN,STATION NAME,CTRY,STATE,ICAO,LAT,LON,ELEV(M),BEGIN,END
23888,843770,99999,CORONEL FRANCISCO SECADA VIGNETTA INTL,PE,NaN,SPQT,-03.785,-073.309,+0093.3,19730101,20250824


In [2]:
est_iquitos = estaciones[estaciones["ICAO"].str.contains("SPQT", na=False)].iloc[0]
est_iquitos

USAF                                            843770
WBAN                                             99999
STATION NAME    CORONEL FRANCISCO SECADA VIGNETTA INTL
CTRY                                                PE
STATE                                              NaN
ICAO                                              SPQT
LAT                                            -03.785
LON                                           -073.309
ELEV(M)                                        +0093.3
BEGIN                                         19730101
END                                           20250824
Name: 23888, dtype: str

In [3]:
estacion_id = str(est_iquitos['USAF']) + str(est_iquitos["WBAN"])
estacion_id

'84377099999'

In [4]:
partes = []

for anio in range(2000, 2026):
    url = f"https://www.ncei.noaa.gov/data/global-hourly/access/{anio}/{estacion_id}.csv"
    try:
        datos_anio = pd.read_csv(url, usecols=["DATE", "TMP", "DEW"], dtype=str)
        partes.append(datos_anio)
        print(f"{anio}: {len(datos_anio)} registros")
    except Exception:
        print(f"{anio}: sin datos")

df = pd.concat(partes, ignore_index=True)
df.to_csv("../data/raw/iquitos_spqt_2000_2025.csv", index=False)

2000: 9014 registros
2001: 9194 registros
2002: 9237 registros
2003: 9383 registros
2004: 9272 registros
2005: 9302 registros
2006: 9665 registros
2007: 9694 registros
2008: 9692 registros
2009: 9377 registros
2010: 9490 registros
2011: 9561 registros
2012: 9832 registros
2013: 11737 registros
2014: 12475 registros
2015: 12507 registros
2016: 12243 registros
2017: 12222 registros
2018: 12035 registros
2019: 12448 registros
2020: 9844 registros
2021: 11868 registros
2022: 12144 registros
2023: 12090 registros
2024: 12058 registros
2025: 8146 registros


In [5]:
df.head(10)

,DATE,TMP,DEW
0,2000-01-01T00:00:00,"+0294,1","+0241,1"
1,2000-01-01T01:00:00,"+0280,1","+0230,1"
2,2000-01-01T02:00:00,"+0280,1","+0240,1"
3,2000-01-01T03:00:00,"+0270,1","+0240,1"
4,2000-01-01T04:00:00,"+0270,1","+0240,1"
5,2000-01-01T05:00:00,"+0250,1","+0240,1"
6,2000-01-01T06:00:00,"+0250,1","+0236,1"
7,2000-01-01T07:00:00,"+0250,1","+0240,1"
8,2000-01-01T08:00:00,"+0240,1","+0230,1"
9,2000-01-01T09:00:00,"+0230,1","+0230,1"


In [6]:
df.info

<bound method DataFrame.info of                        DATE      TMP      DEW
0       2000-01-01T00:00:00  +0294,1  +0241,1
1       2000-01-01T01:00:00  +0280,1  +0230,1
2       2000-01-01T02:00:00  +0280,1  +0240,1
3       2000-01-01T03:00:00  +0270,1  +0240,1
4       2000-01-01T04:00:00  +0270,1  +0240,1
...                     ...      ...      ...
274525  2025-08-24T19:37:00  +0230,2  +0230,1
274526  2025-08-24T19:43:00  +0230,2  +0230,1
274527  2025-08-24T20:00:00  +0240,2  +0230,1
274528  2025-08-24T21:00:00  +0244,1  +0227,1
274529  2025-08-24T21:00:00  +0250,1  +0230,1

[274530 rows x 3 columns]>

In [7]:
df.shape

(274530, 3)

In [8]:
df.describe()

,DATE,TMP,DEW
count,274530,274530,274530
unique,242588,259,193
top,2017-09-08T00:00:00,"+0240,1","+0230,1"
freq,3,47502,79026


Según la informacion encontrada, podemos decir:
- El año 2025 presenta menos datos porque sus datos llegan solo hasta agosto de ese año. Ademas, los años desde el 2000 al 2010 probablemente la estacion no reportaba las 24 horas.
- Los valores que presentan en `TMP` y `DEW` como `+0275` es a temperatura, pero multiplicada por 10 para evitar decimales. Despues de la coma es el código de calidad, indica que pasó los controles de la NOAA. Ádemas, un valor `+9999` significa dato faltante. 


---

## Decodificar la Temperatura

In [9]:
# 1. Separar el valor y el codigo de calidad
df[["tmp_valor", "tmp_calidad"]] = df["TMP"].str.split(",", expand=True)
df[["tmp_valor", "tmp_calidad"]] 

,tmp_valor,tmp_calidad
0,+0294,1
1,+0280,1
2,+0280,1
3,+0270,1
4,+0270,1
...,...,...
274525,+0230,2
274526,+0230,2
274527,+0240,2
274528,+0244,1


In [10]:
# 2. Convertir de texto a numero
df["tmp_valor"] = pd.to_numeric(df["tmp_valor"])
df["tmp_valor"]

0         294
1         280
2         280
3         270
4         270
         ... 
274525    230
274526    230
274527    240
274528    244
274529    250
Name: tmp_valor, Length: 274530, dtype: int64

In [11]:
# 3. Marcar los faltantes como vacios
df.loc[df["tmp_valor"] == 9999, "tmp_valor" ] = None

In [12]:
# 4. Descartar valores sospechosos y/o erroneos
df.loc[df["tmp_calidad"].isin(["2","3","6","7"]), "tmp_valor"] = None

In [13]:
# 5. Converir a grados celcius
df["tmp_valor"] = df["tmp_valor"] / 10
df["tmp_valor"]

0         29.4
1         28.0
2         28.0
3         27.0
4         27.0
          ... 
274525     NaN
274526     NaN
274527     NaN
274528    24.4
274529    25.0
Name: tmp_valor, Length: 274530, dtype: float64

---

# Decodificar el Punto de Rocio (DWE)

In [46]:
df["DEW"]

0         +0241,1
1         +0230,1
2         +0240,1
3         +0240,1
4         +0240,1
           ...   
274525    +0230,1
274526    +0230,1
274527    +0230,1
274528    +0227,1
274529    +0230,1
Name: DEW, Length: 274530, dtype: str

In [47]:
df[["tmp_rocio", "punto_rocio"]] = df["DEW"].str.split(",", expand=True)
df[["tmp_rocio", "punto_rocio"]]

,tmp_rocio,punto_rocio
0,+0241,1
1,+0230,1
2,+0240,1
3,+0240,1
4,+0240,1
...,...,...
274525,+0230,1
274526,+0230,1
274527,+0230,1
274528,+0227,1


In [48]:
df["tmp_rocio"] = pd.to_numeric(df["tmp_rocio"])
df["tmp_rocio"]

0         241
1         230
2         240
3         240
4         240
         ... 
274525    230
274526    230
274527    230
274528    227
274529    230
Name: tmp_rocio, Length: 274530, dtype: int64

In [49]:
df.loc[df["tmp_rocio"] == 9999, "tmp_rocio"] = None

In [50]:
df.loc[df["punto_rocio"].isin(["2","3","6","7"]), "punto_rocio"] = None

In [51]:
df["tmp_rocio"] =  df["tmp_rocio"] / 10
df["tmp_rocio"]

0         24.1
1         23.0
2         24.0
3         24.0
4         24.0
          ... 
274525    23.0
274526    23.0
274527    23.0
274528    22.7
274529    23.0
Name: tmp_rocio, Length: 274530, dtype: float64